In [ ]:
import pandas as pd
from datetime import datetime


In [ ]:
# Load CSV files
positive_cases = pd.read_csv('POSITIVE_GROUP_FINAL.csv', parse_dates=['study_date_anon', 'diagnosed_date_anon'])
negative_cases = pd.read_csv('NEGATIVE_GROUP_FINAL.csv', parse_dates=['study_date_anon'])


In [ ]:
negative_cases.rename(columns={'empi_anon': 'patient_id', 'acc_anon': 'exam_id', 'ViewPosition':'view'}, inplace=True)


In [ ]:
display(positive_cases)

In [ ]:
display(negative_cases)

In [ ]:
# For positive cases
def calculate_positive_followup(row):
    # Compute difference in years between the exam date and cancer diagnosis date
    return (row['diagnosed_date_anon'] - row['study_date_anon']).days / 365.25

positive_cases['years_last_followup'] = positive_cases.apply(calculate_positive_followup, axis=1)
positive_cases['years_last_followup'] = positive_cases['years_last_followup'].astype(int)
# For negative cases
def calculate_negative_followup(patient_df):
    # Determine the latest exam date for each patient
    latest_date = patient_df['study_date_anon'].max()
    # Compute difference in years between the exam date and the latest available date
    patient_df['years_last_followup'] = (latest_date - patient_df['study_date_anon']).dt.days / 365.25
    patient_df['years_last_followup'] = patient_df['years_last_followup'].astype(int)
    return patient_df


In [ ]:
display(positive_cases)

In [ ]:

negative_cases = negative_cases.groupby('patient_id').apply(calculate_negative_followup)

# Combine datasets if needed
all_cases = pd.concat([positive_cases, negative_cases], ignore_index=True)



In [ ]:
display(all_cases)

In [ ]:
# Save to CSV
all_cases.to_csv('combined_cases_with_followup.csv', index=False)